# Feature Engineering

In [63]:
import pandas as pd
import numpy as np

In [64]:
data = pd.read_csv(
    "../Data/Processed/solar_prepared.csv"
)

data["time_sl"] = (
    pd.to_datetime(
        data["time_sl"],
        utc=True
    )
    .dt.tz_convert("Asia/Colombo")
)

print("Dataset shape:", data.shape)

Dataset shape: (437975, 26)


In [65]:
data["time_utc"] = pd.to_datetime(
    data["time_utc"],
    utc=True
)

Check Solar Active Periods

In [66]:
pd.crosstab(
    data["global_tilted_irradiance"] > 0,
    data["P"] > 0,
    rownames=["GTI > 0"],
    colnames=["P > 0"]
)

P > 0,False,True
GTI > 0,,
False,206178,24
True,12507,219266


In [67]:
gti_zero = data[
    data["global_tilted_irradiance"] == 0
]

print(
    "Rows with GTI = 0:",
    len(gti_zero)
)

print(
    "Positive P when GTI = 0:",
    (gti_zero["P"] > 0).sum()
)

print(
    "Maximum P when GTI = 0:",
    gti_zero["P"].max()
)

Rows with GTI = 0: 206202
Positive P when GTI = 0: 24
Maximum P when GTI = 0: 8.56


In [68]:
p_zero = data[
    data["P"] == 0
]

print(
    "Positive GTI when P = 0:",
    (
        p_zero["global_tilted_irradiance"] > 0
    ).sum()
)

print(
    "Maximum GTI when P = 0:",
    p_zero[
        "global_tilted_irradiance"
    ].max()
)

Positive GTI when P = 0: 12507
Maximum GTI when P = 0: 519.3


In [69]:
low_gti = data[
    (
        data["global_tilted_irradiance"] > 0
    )
    &
    (
        data["global_tilted_irradiance"] <= 20
    )
]

print("Rows with GTI 0-20:", len(low_gti))

print(
    "Average P:",
    low_gti["P"].mean()
)

print(
    "Maximum P:",
    low_gti["P"].max()
)

Rows with GTI 0-20: 23843
Average P: 7.60136392232521
Maximum P: 527.29


Create Solar Active Dataset

In [70]:
model_data = data[
    data["global_tilted_irradiance"] > 0
].copy()

print("Original rows:", len(data))
print("Solar active rows:", len(model_data))
print(
    "Removed GTI zero rows:",
    len(data) - len(model_data)
)

Original rows: 437975
Solar active rows: 231773
Removed GTI zero rows: 206202


 Handle Missing Precipitation

In [71]:
print(
    "Missing precipitation before removal:",
    model_data["precipitation"].isnull().sum()
)

model_data = model_data.dropna(
    subset=["precipitation"]
).copy()

print(
    "Rows after removing missing precipitation:",
    len(model_data)
)

print(
    "Missing precipitation after removal:",
    model_data["precipitation"].isnull().sum()
)

Missing precipitation before removal: 4555
Rows after removing missing precipitation: 227218
Missing precipitation after removal: 0


Create Time Features

In [72]:
model_data["hour"] = (
    model_data["time_sl"].dt.hour
)

model_data["day_of_year"] = (
    model_data["time_sl"].dt.dayofyear
)

In [73]:
model_data["hour_sin"] = np.sin(
    2 * np.pi * model_data["hour"] / 24
)

model_data["hour_cos"] = np.cos(
    2 * np.pi * model_data["hour"] / 24
)

model_data["day_of_year_sin"] = np.sin(
    2 * np.pi
    * model_data["day_of_year"]
    / 365.25
)

model_data["day_of_year_cos"] = np.cos(
    2 * np.pi
    * model_data["day_of_year"]
    / 365.25
)

Check Time Features

In [74]:
model_data[
    [
        "time_sl",
        "hour",
        "hour_sin",
        "hour_cos",
        "day_of_year",
        "day_of_year_sin",
        "day_of_year_cos"
    ]
].head()

,time_sl,hour,hour_sin,hour_cos,day_of_year,day_of_year_sin,day_of_year_cos
1,2022-01-01 07:30:00+05:30,7,0.965926,-0.258819,1,0.017202,0.999852
2,2022-01-01 08:30:00+05:30,8,0.866025,-0.500000,1,0.017202,0.999852
3,2022-01-01 09:30:00+05:30,9,0.707107,-0.707107,1,0.017202,0.999852
4,2022-01-01 10:30:00+05:30,10,0.500000,-0.866025,1,0.017202,0.999852
5,2022-01-01 11:30:00+05:30,11,0.258819,-0.965926,1,0.017202,0.999852


In [75]:
print(
    "Hour range:",
    model_data["hour"].min(),
    "to",
    model_data["hour"].max()
)

print(
    "Day of year range:",
    model_data["day_of_year"].min(),
    "to",
    model_data["day_of_year"].max()
)

Hour range: 6 to 18
Day of year range: 1 to 365


Create Weather Lag Features

In [76]:
data = data.sort_values(
    ["district", "time_utc"]
).copy()

data["gti_lag_1"] = (
    data.groupby("district")[
        "global_tilted_irradiance"
    ]
    .shift(1)
)

data["cloud_cover_lag_1"] = (
    data.groupby("district")[
        "cloud_cover"
    ]
    .shift(1)
)

In [77]:
model_data["gti_lag_1"] = data.loc[
    model_data.index,
    "gti_lag_1"
]

model_data["cloud_cover_lag_1"] = data.loc[
    model_data.index,
    "cloud_cover_lag_1"
]

In [78]:
print(
    "Missing GTI lag:",
    model_data["gti_lag_1"].isnull().sum()
)

print(
    "Missing cloud lag:",
    model_data["cloud_cover_lag_1"].isnull().sum()
)

Missing GTI lag: 0
Missing cloud lag: 0


Define Model Features

In [79]:
feature_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "global_tilted_irradiance",
    "diffuse_radiation",
    "sunshine_duration",
    "latitude",
    "longitude",
    "elevation",
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "gti_lag_1",
    "cloud_cover_lag_1"
]

target_column = "P"

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 17
['temperature_2m', 'relative_humidity_2m', 'precipitation', 'cloud_cover', 'wind_speed_10m', 'global_tilted_irradiance', 'diffuse_radiation', 'sunshine_duration', 'latitude', 'longitude', 'elevation', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'gti_lag_1', 'cloud_cover_lag_1']


Create Modelling Dataset

In [80]:
model_columns = (
    [
        "time_utc",
        "time_sl",
        "district"
    ]
    + feature_columns
    + [target_column]
)

final_model_data = model_data[
    model_columns
].copy()

print(
    "Modelling dataset shape:",
    final_model_data.shape
)

Modelling dataset shape: (227218, 21)


Dataset Quality

In [81]:
print("Missing values:")

print(
    final_model_data
    .isnull()
    .sum()[
        final_model_data.isnull().sum() > 0
    ]
)

print(
    "\nDuplicate district/time:",
    final_model_data.duplicated(
        subset=["district", "time_utc"]
    ).sum()
)

Missing values:
Series([], dtype: int64)

Duplicate district/time: 0


In [82]:
numeric_columns = (
    feature_columns + [target_column]
)

print(
    "Infinite values:",
    np.isinf(
        final_model_data[numeric_columns]
    ).sum().sum()
)

Infinite values: 0


 Check Modelling Time Range

In [83]:
print(
    "First timestamp:",
    final_model_data["time_sl"].min()
)

print(
    "Last timestamp:",
    final_model_data["time_sl"].max()
)

print(
    "Districts:",
    final_model_data["district"].nunique()
)

First timestamp: 2022-01-01 07:30:00+05:30
Last timestamp: 2023-12-31 18:30:00+05:30
Districts: 25


Check Feature Ranges

In [84]:
final_model_data[
    feature_columns + ["P"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
temperature_2m,227218.0,27.967192,3.851299,8.600000,25.900000,2.810000e+01,30.300000,4.230000e+01
relative_humidity_2m,227218.0,69.600903,15.613868,14.000000,59.000000,7.100000e+01,82.000000,1.000000e+02
precipitation,227218.0,0.277485,0.821536,0.000000,0.000000,0.000000e+00,0.100000,6.000000e+01
cloud_cover,227218.0,75.053332,35.277527,0.000000,49.000000,1.000000e+02,100.000000,1.000000e+02
wind_speed_10m,227218.0,12.157272,7.384196,0.000000,6.300000,1.090000e+01,16.600000,5.110000e+01
global_tilted_irradiance,227218.0,446.695117,325.888710,0.100000,131.000000,4.213000e+02,751.800000,1.149100e+03
diffuse_radiation,227218.0,139.343415,95.170667,0.000000,65.000000,1.310000e+02,202.000000,4.140000e+02
sunshine_duration,227218.0,2966.626410,1237.988835,0.000000,3555.757500,3.600000e+03,3600.000000,3.600000e+03
latitude,227218.0,7.585213,1.036366,5.948510,6.935480,7.297540e+00,8.312230,9.668450e+00
longitude,227218.0,80.586332,0.538330,79.828300,80.210300,8.049710e+01,81.002740,8.169240e+01


Saving Feature-Engineered Dataset

In [85]:
output_file = (
    "../Data/Processed/"
    "solar_model_data.csv"
)

final_model_data.to_csv(
    output_file,
    index=False
)

print(
    "Saved modelling dataset to:",
    output_file
)

Saved modelling dataset to: ../Data/Processed/solar_model_data.csv
